In [ ]:
import numpy as np
import polars as pl

import seaborn as sns

from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset
from climate_attitudes.utils import (
    calculate_proportion_response,
    calculate_stationary_distribution,
    calculate_transition_probabilities,
    relative_entropy,
)

config = Config(_env_file="../.env")

data = Dataset.load(config)

Filter down to participants who are present in waves 1 and 2

In [ ]:
pids = (
    data.participant.filter(wave_1=True, wave_2=True)
    .select("participant_id")
    .collect()
    .to_series()
    .implode()
)

resp = (
    data.response.filter(pl.col("wave") <= 2, pl.col("participant_id").is_in(pids))
    .select(
        "participant_id",
        "wave",
        # pl.col("wave").replace_strict({1: "wave_1", 2: "wave_2"}),
        pl.col("cc1").replace({1: 2, 99: 1}),
        "cvcc4_should",
        pl.col("cc5_world").replace(99, 0),
        "cc6",
    )
    .with_columns(pl.len().over("participant_id").alias("n_waves"))
    .filter(n_waves=2)
    .collect()
)

columns = {
    "Climate change happening": {
        "colname": "cc1",
        "responses": ["No", "Don't know", "Yes"],
    },
    "Climate change anthropogenic ('people should act')": {
        "colname": "cvcc4_should",
        "responses": [
            "Strongly disagree",
            "Disagree",
            "Ambivalent",
            "Agree",
            "Strongly agree",
        ],
    },
    "Climate change worry": {
        "colname": "cc6",
        "responses": [
            "Not at all worried",
            "Not very worried",
            "Somewhat worried",
            "Very worried",
        ],
    },
    "Future generation harm": {
        "colname": "cc5_world",
        "responses": [
            "Don't know",
            "Not at all",
            "Only a little",
            "A moderate amount",
            "A great deal",
        ],
    },
}

Calculate relative entropy between Wave 1 distribution and stationary distribution

In [ ]:
plot_data = {"Question": [], "Relative entropy": []}

for i, (dimension, dimension_metadata) in enumerate(columns.items()):
    colname = dimension_metadata["colname"]

    transition_probabilities = calculate_transition_probabilities(
        resp, column=colname, start=1, end=2
    )

    if dimension_metadata["colname"].startswith("cc5_"):
        transition_probabilities = np.hstack(
            (np.zeros(5)[:, None], transition_probabilities)
        )

    # Calculate stationary distribution
    mu = calculate_stationary_distribution(transition_probabilities)

    # Calculate wave 1 distribution
    props = calculate_proportion_response(resp, column=colname, wave=1)

    # Calculate relative entropy (expected excess surprise when observing true props)
    plot_data["Question"].append(colname)
    plot_data["Relative entropy"].append(relative_entropy(props, mu))

plot_df = pl.DataFrame(plot_data)

In [ ]:
sns.catplot(plot_df, x="Question", y="Relative entropy", kind="bar")